In [ ]:
import yfinance as yf
import pandas as pd
import openai
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt

# Cargar clave de OpenAI (opcional: usar .env)
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

# Lista de 30 acciones (¡reemplaza con las tuyas!)
tickers = [
    "AZN", "PFE", "MRNA", "JNJ", "ABBV", "LLY", "UNH", "JPM", "MSFT", "AAPL",
    "TSLA", "NVDA", "AMD", "INTC", "CRM", "ADBE", "GOOGL", "AMZN", "META", "NFLX",
    "DIS", "V", "PG", "KO", "PEP", "MMM", "BA", "CAT", "GE", "MMM"
]

# --- 1. Función para obtener datos fundamentales y técnicos ---
def get_stock_data(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    hist = stock.history(period="1y")

    # Datos fundamentales
    pe = info.get('trailingPE', None)
    pb = info.get('priceToBook', None)
    debt_equity = info.get('debtToEquity', None)
    roe = info.get('returnOnEquity', None)
    revenue_growth = info.get('revenueGrowth', None)
    eps_growth = info.get('earningsGrowth', None)
    net_margin = info.get('netProfitMargin', None)
    free_cash_flow = info.get('freeCashflow', None)
    dividend_yield = info.get('dividendYield', None)
    beta = info.get('beta', None)

    # Datos técnicos (últimos 14 días)
    if len(hist) >= 14:
        atr = hist['Close'].rolling(window=14).std() * 1.96  # Aproximación simple
        rsi = calculate_rsi(hist['Close'], 14)
        macd, signal = calculate_macd(hist['Close'], 12, 26, 9)
        macd_hist = macd - signal
    else:
        atr = None
        rsi = None
        macd = None
        macd_hist = None

    # Volumen y liquidez
    volume = hist['Volume'].iloc[-1] if len(hist) > 0 else None
    avg_volume = hist['Volume'].mean() if len(hist) > 0 else None

    return {
        'Ticker': ticker,
        'Price': hist['Close'].iloc[-1] if len(hist) > 0 else None,
        'P/E': pe,
        'P/B': pb,
        'Debt/Equity': debt_equity,
        'ROE': roe,
        'Revenue Growth YoY': revenue_growth,
        'EPS Growth YoY': eps_growth,
        'Net Margin': net_margin,
        'Free Cash Flow': free_cash_flow,
        'Dividend Yield': dividend_yield,
        'Beta': beta,
        'ATR': atr,
        'RSI': rsi,
        'MACD': macd,
        'MACD Histogram': macd_hist,
        'Volume': volume,
        'Avg Volume': avg_volume,
        'Sentiment': "N/A",  # IA lo analizará
        'Short Interest': "N/A",
        'Earnings Surprise': "N/A",
        'Regulatory Risk': "N/A",
        'Growth Strategy': "N/A"
    }

# --- 2. Función para calcular RSI ---
def calculate_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi.iloc[-1] if not rsi.empty else None

# --- 3. Función para calcular MACD ---
def calculate_macd(series, fast=12, slow=26, signal=9):
    exp1 = series.ewm(span=fast).mean()
    exp2 = series.ewm(span=slow).mean()
    macd = exp1 - exp2
    signal_line = macd.ewm(span=signal).mean()
    return macd.iloc[-1], signal_line.iloc[-1]

# --- 4. Prompt de IA para análisis ---
def generate_ai_analysis(ticker_data):
    prompt = f"""
    Analiza la acción de {ticker_data['Ticker']} usando las siguientes métricas:
    - Precio actual: {ticker_data['Price']}
    - P/E: {ticker_data['P/E']}
    - P/B: {ticker_data['P/B']}
    - Deuda/Equity: {ticker_data['Debt/Equity']}
    - ROE: {ticker_data['ROE']}
    - Crecimiento de ingresos YoY: {ticker_data['Revenue Growth YoY']}
    - Crecimiento de ganancias YoY: {ticker_data['EPS Growth YoY']}
    - Margen neto: {ticker_data['Net Margin']}
    - Flujo de efectivo libre: {ticker_data['Free Cash Flow']}
    - Dividendo y yield: {ticker_data['Dividend Yield']}
    - Beta: {ticker_data['Beta']}
    - Volatilidad (ATR): {ticker_data['ATR']}
    - RSI: {ticker_data['RSI']}
    - MACD: {ticker_data['MACD']}
    - MACD Histogram: {ticker_data['MACD Histogram']}
    - Volumen: {ticker_data['Volume']}
    - Avg Volumen: {ticker_data['Avg Volume']}
    - Sentimiento del mercado: N/A (analiza noticias y redes)
    - Short Interest: N/A
    - Earnings Surprise: N/A
    - Riesgo regulatorio: N/A
    - Estrategia de crecimiento: N/A

    Basado en estos datos, asigna:
    - Una recomendación de 1 a 5 estrellas (1 = no comprar, 5 = excelente oportunidad)
    - Un score final de 0 a 100 (basado en: riesgo relativo 20%, potencial retorno 30%, consistencia histórica 20%, liquidez 15%, valoración fundamental 15%, sentimiento y riesgo externo 10%)
    - Una explicación breve de por qué es buena o mala
    - Una estrategia de entrada (stop loss, take profit) y tamaño de posición (ej: 2-5% del capital por acción)
    - Indica si es defensiva, agresiva o equilibrada

    Devuelve solo el análisis en formato JSON con las claves: 
    {{
        "recomendation_stars": int,
        "final_score": float,
        "reason": str,
        "entry_strategy": str,
        "risk_level": str  # "Defensiva", "Equilibrada", "Agresiva"
    }}
    """
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=1000
        )
        content = response.choices[0].message.content
        # Parsear JSON (simplificado, en producción usa json.loads)
        import re
        score_match = re.search(r'"final_score":\s*([\d.]+)', content)
        stars_match = re.search(r'"recomendation_stars":\s*(\d)', content)
        reason_match = re.search(r'"reason":\s*"([^"]*)"', content)
        entry_match = re.search(r'"entry_strategy":\s*"([^"]*)"', content)
        risk_match = re.search(r'"risk_level":\s*"([^"]*)"', content)

        return {
            "recomendation_stars": int(stars_match.group(1)) if stars_match else 0,
            "final_score": float(score_match.group(1)) if score_match else 0.0,
            "reason": reason_match.group(1) if reason_match else "No se pudo analizar",
            "entry_strategy": entry_match.group(1) if entry_match else "No se pudo determinar",
            "risk_level": risk_match.group(1) if risk_match else "Desconocido"
        }
    except Exception as e:
        print(f"Error al llamar a la IA para {ticker_data['Ticker']}: {e}")
        return {
            "recomendation_stars": 0,
            "final_score": 0.0,
            "reason": "Error en IA",
            "entry_strategy": "No se pudo determinar",
            "risk_level": "Error"
        }

# --- 5. Función principal ---
def main():
    print("🔍 Analizando 30 acciones con sistema de TradeSpider...")
    results = []

    for ticker in tickers:
        print(f"📊 Procesando: {ticker}...")
        data = get_stock_data(ticker)
        ai_analysis = generate_ai_analysis(data)
        data.update(ai_analysis)
        results.append(data)

    # Crear DataFrame
    df = pd.DataFrame(results)

    # Guardar en Excel
    df.to_excel("analisis_stock_trade_spider.xlsx", index=False)
    print("\n✅ Resultados guardados en 'analisis_stock_trade_spider.xlsx'")

    # Mostrar las 5 mejores
    df_sorted = df.sort_values(by='final_score', ascending=False).head(5)
    print("\n🏆 Las 5 mejores acciones:")
    print(df_sorted[['Ticker', 'final_score', 'recomendation_stars', 'reason', 'risk_level', 'entry_strategy']])

    # Gráfico de scores
    plt.figure(figsize=(10, 6))
    plt.bar(df_sorted['Ticker'], df_sorted['final_score'], color='lightgreen')
    plt.title("Score Final de las 5 mejores acciones")
    plt.xlabel("Acción")
    plt.ylabel("Score (0-100)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("scores_top5.png")
    plt.show()

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'yfinance'